This workflow provides a complete working example to develop an unstructured mesh for an integrated hydrologic model for a 2D transect hillslope. is revised from the following two notebooks:
- [2D transect simple meshing example](https://github.com/amanzi/ats-short-course/blob/main/02_model_setup/simple_meshing.ipynb)
- [Complete Workflow for generating ATS input for Coweeta](https://github.com/environmental-modeling-workflows/watershed-workflow/blob/v1.5/examples/Coweeta/coweeta_ats.ipynb)

It uses the following datasets:
- NHD Plus for the watershed boundary and hydrography.
- NED for elevation
- NLCD for land cover/transpiration/rooting depths
- GLYHMPS geology data for structural formations
- SoilGrids 2017 for depth to bedrock and soil texture information
- SSURGO for soil data, where available, in the top 2m.
- NRCS soil survey data, for soil structure.


**File History**

update 2025/10/13
- update `config.json`. Mainly revise the model run pipeline.

update 2025/8/32
- add `config.json`
- separate hillslope Daymet/MODIS/BCHead extraction to individual notebooks
    - `get_Daymet.ipynb`
    - `get_MODIS-LAI_OakCreek.ipynb`
    - `get_BChead.ipynb`

update 2025/8/7
- load start_coords and end_coords from mat file generated by "0-transect_latlon.NF01.ipynb".
    - data type is numpy.float64, so precision is double.
- calculate boundary head for steady state spinup from "data-processed/NF01/startpt_head_10y_typical.h5 and endpt_head_10y_typical.h5"
    - BC head typical year is output from notebook "notebooks/get_BChead_OakCreek.ipynb"
- complete and test the xml generation part

update 2025/5/21
- revise watershed_workflow.condition.fill_pits(m2, outlet=meshsize_nx, algorithm=3). Outlet by default = None, and the fill_pits algorithm will automatically find the lowest points
- save m2 to mat file, for later usage in "2-obs_for_boundary_mass_flux.ipynb"

update 2025/4/23
- revise configuration to start_year/end_year and startdate/enddate
- revise "time [s]" in DayMet data for run1 and run2
- revise "time [s]" in MODIS-LAI data for run1 and run2
- in this version, run1 and run2 will actually start and end Oct1 of the year.
- DayMet and MODIS-LAI data start and end Jan1

updated 2025/1/22
- compatible with watershed_workflow v1.5 and ats_input_spec v1.5

In [ ]:
%load_ext autoreload
%autoreload 2

# Parameters and data sources

In [ ]:
# Parameters cell
import json
with open('config.json', 'r') as f:
    config = json.load(f)
watershed_name = config['watershed_name']
hucs           = [config['hucs']]
site_name      = config['site_name']

# simulation control
start_year_spinup         = config['start_year_spinup']
end_year_spinup           = config['end_year_spinup']
nyears_steadystate_spinup = config['nyears_steadystate_spinup']
nyears_cyclic_spinup      = config['nyears_cyclic_spinup']
start_year_transient      = config['start_year_transient']
end_year_transient        = config['end_year_transient']

# logistics
# generate_plots = True # plots take time to make and aren't always needed
# generate_daymet = True # potentially don't do Met data forcing
# generate_modis = True

# include_heterogeneous = True
# include_homogeneous = False # if true, also write files for homogeneous runs
# include_homogeneous_wrm = False # if true, also write files for homogeneous WRMs
# include_homogeneous_wrm_porosity = False # if true, also write files for homogeneous porosity and WRMs
# include_homogeneous_wrm_permeability = False # if true, also write files for homogeneous perm and WRMs

# log_to_file = False  # if true, write to file instead of in the notebook output

In [ ]:
# parameter checking
#assert(simplify > 0 and simplify < 300)
#assert(ignore_small_rivers == None or (ignore_small_rivers >= 0 and ignore_small_rivers <= 100))
#assert(prune_by_area_fraction == None or (prune_by_area_fraction >= 0 and prune_by_area_fraction < 1))
assert(start_year_spinup >= 1980 and end_year_transient < 2024)

In [ ]:
# a dictionary of outputs -- will include all filenames generated
outputs = {}

In [ ]:
# conda package imports
import logging
import sys,os
# sys.path.append(os.path.join(os.environ['ATS_SRC_DIR'],'tools','meshing','meshing_ats'))
# import meshing_ats


import geopandas as gpd
import numpy as np
import pandas as pd
pd.options.display.max_columns = None
pd.options.display.max_rows = 20
import h5py as h5
import rasterio
import shapely
from shapely.geometry import Point, LineString, Polygon, box, mapping
import fiona
import cftime, datetime
import scipy.ndimage
from scipy.io import loadmat
from scipy.io import savemat

import matplotlib
import matplotlib.colors as colors
from matplotlib import pyplot as plt

# Watershed Workflow
import watershed_workflow
import watershed_workflow.source_list
import watershed_workflow.ui
import watershed_workflow.colors
import watershed_workflow.condition
import watershed_workflow.mesh
import watershed_workflow.split_hucs
import watershed_workflow.soil_properties
import watershed_workflow.daymet
import watershed_workflow.utils
import watershed_workflow.regions


# ats_input_spec library, to be moved to amanzi_xml
import ats_input_spec
import ats_input_spec.public
import ats_input_spec.io
from ats_input_spec.public import known_specs

# amanzi_xml, included in AMANZI_SRC_DIR/tools/amanzi_xml
import amanzi_xml.utils.io as aio
import amanzi_xml.utils.search as asearch
import amanzi_xml.utils.errors as aerrors
from amanzi_xml.common.parameter import Parameter
from amanzi_xml.common.parameter_list import ParameterList

In [ ]:
# Note that, by default, we tend to work in the DayMet CRS because this allows us to avoid
# reprojecting meteorological forcing datasets.
crs_daymet = watershed_workflow.crs.daymet_crs()
crs_daymet

In [ ]:
# Plotting parameters
# rc('text', usetex=False)
small_size = 15
medium_size = 25
bigger_size = 30
plt.rc('font', size=small_size)          # controls default text sizes
plt.rc('axes', titlesize=small_size)    # fontsize of the axes title
plt.rc('axes', labelsize=small_size)    # fontsize of the x and y labels
plt.rc('xtick', labelsize=small_size)    # fontsize of the tick labels
plt.rc('ytick', labelsize=small_size)    # fontsize of the tick labels
plt.rc('legend', fontsize=small_size)    # legend fontsize
plt.rc('figure', titlesize=small_size)  # fontsize of the figure title
plt.rc('text', usetex = False)

In [ ]:
# set up a dictionary of source objects
sources = watershed_workflow.source_list.get_default_sources()
sources['hydrography'] = watershed_workflow.source_list.hydrography_sources['NHD Plus']
sources['HUC'] = watershed_workflow.source_list.huc_sources['NHD Plus']
sources['DEM'] = watershed_workflow.source_list.dem_sources['NED 1/3 arc-second']
sources['geologic structure'] = watershed_workflow.source_list.FileManagerGLHYMPS('./data/soil_structure/GLHYMPS/GLHYMPS.shp')
sources['depth to bedrock'] = watershed_workflow.source_list.FileManagerRaster('./data/soil_structure/SoilGrids2017/BDTICM_M_250m_ll.tif')
watershed_workflow.source_list.log_sources(sources)
sources

In [ ]:
# Prepare '../data-processed' folders
os.makedirs(f'../data-processed/{watershed_name}', exist_ok=True)
os.makedirs(f'../data-processed/{site_name}', exist_ok=True)

# Prepare './images' folders
os.makedirs(f'./images/{site_name}', exist_ok=True)
os.makedirs(f'./images/{watershed_name}', exist_ok=True)

# Generate watershed boundary

In [ ]:
#watershed_name = 'OakCreek' # name the domain, used in filenames, etc
#hucs = ['170300020307'] # a list of HUCs to run
def get_huc12(hucs):
    huc12_list = []
    for huc in hucs:
        if len(huc) == 12:
            huc12_list.append(huc)
        elif len(huc) == 10:
            for i in range(1,20):
                huc12_list.append(huc+str(i).zfill(2))
        elif len(huc) == 8:
            for i in range(1,20):
                for j in range(1,20):
                    huc12_list.append(huc+str(i).zfill(2)+str(j).zfill(2))
        else:
            print('need huc8, huc10 or huc12')
    return huc12_list

hucs = get_huc12(hucs)
print(hucs[:10])

In [ ]:
huc_level = 12 # if provided, an int setting the level at which to include HUC boundaries

In [ ]:
my_hucs = []
for huc in hucs:
    _, ws = watershed_workflow.get_hucs(sources['HUC'], huc, huc_level, crs_daymet)
    my_hucs.extend(ws)

watershed = watershed_workflow.split_hucs.SplitHUCs(my_hucs)

In [ ]:
### save watershed boundary shapefile
# try:
#     os.mkdir(f'../data-processed/{watershed_name}')
# except FileExistsError:
#     pass

outputs['watershed_shapefile_filename'] = f'../data-processed/{watershed_name}/{watershed_name}_bounds.shp'
# Define a polygon feature geometry with one attribute
schema = {
    'geometry': 'Polygon',
    'properties': {'id': 'int'},
}

# Write a new Shapefile
with fiona.open(outputs['watershed_shapefile_filename'], 'w', 'ESRI Shapefile', schema, crs=watershed_workflow.crs.to_fiona(crs_daymet)) as c:
    ## If there are multiple geometries, put the "for" loop here
    c.write({
        'geometry': mapping(watershed.exterior()),
        'properties': {'id': 123},
    })

# Generate surface mesh

## mesh segment and find elevation

In [ ]:
# Function for extracting elevations of the cross section
# 2025/01/24 by Yi Xiao
# customized for input tif image generated by Watershed-Workflow, with crs converted to Daymet and saved as reprojected_dem.tif
def get_xsec_elevation_3(start_coords, end_coords, dem_file, npts=10):
    # Read the DEM file
    dem = rasterio.open(dem_file)
    
    # Get the list of coordinates in the x-section
    lon, lat = [start_coords[0]], [start_coords[1]]
    x_dist = end_coords[0] - start_coords[0]
    y_dist = end_coords[1] - start_coords[1]
    for i in np.arange(1, npts+1):
        point  = [(start_coords[0]+(x_dist/(npts+1))*i), (start_coords[1]+(y_dist/(npts+1))*i)]
        lon.append(point[0])
        lat.append(point[1])
    lon.append(end_coords[0])
    lat.append(end_coords[1])
    
    df  = pd.DataFrame({"lon":lon, "lat":lat})
    gdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.lon, df.lat))
    gdf.crs = dem.crs.data
    
    # Compute the distance between points
    gdf['h_distance'] = 0.0
    for index, row in gdf.iterrows():
        #gdf['h_distance'].loc[index] = gdf.geometry[0].distance(gdf.geometry[index])
        gdf.loc[index, 'h_distance'] = gdf.geometry[0].distance(gdf.geometry[index])
        
    # Get the elevation from DEM
    gdf['elevation'] = 0.0
    for index, row in gdf.iterrows():
        row, col = dem.index(row['lon'], row['lat'])
        #gdf['elevation'].loc[index] = dem.read(1)[row, col]
        gdf.loc[index, 'elevation'] = dem.read(1)[row, col]
    
    dem.close()
    
    return gdf.round({'h_distance':3, 'elevation':3})

In [ ]:
# import pyproj

# # lon lat to x y
# wgs84 = pyproj.Proj(proj='latlong', datum='WGS84')
# proj_daymet = pyproj.Proj('+proj=lcc +lat_1=25 +lat_2=60 +lat_0=42.5 +lon_0=-100 +x_0=0 +y_0=0 +ellps=WGS84 +units=m +no_defs')
# ## NF01, M01, M02, M03, SF01
# lat1, lon1 = 46.742847, -120.92965
# lat2, lon2 = 46.72375 , -120.81516
# lat3, lon3 = 46.729449, -120.9368
# lat4, lon4 = 46.716332, -121.00993
# lat5, lon5 = 46.728668, -120.93757

# x1, y1 = pyproj.transform(wgs84, proj_daymet, lon1, lat1)
# x2, y2 = pyproj.transform(wgs84, proj_daymet, lon2, lat2)
# x3, y3 = pyproj.transform(wgs84, proj_daymet, lon3, lat3)
# x4, y4 = pyproj.transform(wgs84, proj_daymet, lon4, lat4)
# x5, y5 = pyproj.transform(wgs84, proj_daymet, lon5, lat5)

# print(f"Longitude: {lon1}, Latitude: {lat1}")
# print(f"X: {x1}, Y: {y1}")
# print(f"Longitude: {lon2}, Latitude: {lat2}")
# print(f"X: {x2}, Y: {y2}")
# print(f"Longitude: {lon3}, Latitude: {lat3}")
# print(f"X: {x3}, Y: {y3}")
# print(f"Longitude: {lon4}, Latitude: {lat4}")
# print(f"X: {x4}, Y: {y4}")

# site1_xy = shapely.geometry.Point(x1, y1)
# site2_xy = shapely.geometry.Point(x2, y2)
# site3_xy = shapely.geometry.Point(x3, y3)
# site4_xy = shapely.geometry.Point(x4, y4)
# site5_xy = shapely.geometry.Point(x5, y5)

In [ ]:
# Reset logging configuration
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

watershed_workflow.ui.setup_logging(1,None)

logging.info("")
logging.info(f"Meshing a 2D transect for site NF01")
logging.info("="*30)

In [ ]:
# DEM file
f_dem = './data/dem/reprojected_dem.tif'

# Start and end locations for getting the 2d transect
# site NF01; along perpendicular direction [see 0-transect_latlon.ipynb]
#start_coords, end_coords = (-1511015.74507677, 640547.2830956738), (-1511677.2648839361, 640389.829786001) # try to change from 440450 to 440850_BL

m2_mat_filename =  f'../data-processed/{site_name}/startendcoords_{site_name}.mat'
loaded_data  = loadmat(m2_mat_filename)
start_coords = loaded_data['start_coords'].flatten()
end_coords   = loaded_data['end_coords'].flatten()

gdf = get_xsec_elevation_3(start_coords, end_coords, f_dem, npts=250) # gw 1 - gw 8
xsec = LineString([start_coords, end_coords])

In [ ]:
# Visualize the 2D transect in DEM
# site NF01
dx = 5000/2
dy = 4000/2
xmin = (start_coords[0]+end_coords[0])/2 - dx/2
xmax = (start_coords[0]+end_coords[0])/2 + dx/2
ymin = (start_coords[1]+end_coords[1])/2 - dy/2
ymax = (start_coords[1]+end_coords[1])/2 + dy/2

#norm = colors.TwoSlopeNorm(vmin=-2, vcenter=0, vmax=20)
with rasterio.open(f_dem) as dem:
    cmap = matplotlib.cm.jet
    cmap.set_bad(color='black')
#     print(dem.bounds, dem.block_shapes)
    l,b,r,t = dem.bounds
    raster = dem.read(1).copy()
    mask = dem.read_masks(1).copy()
    raster[mask==0] = np.nan
    fig, ax = plt.subplots(1,1,figsize=(10,8))
    im=ax.imshow(raster, extent=[l,r,b,t], cmap='terrain')
#     gpr_box_gdf.plot(ax=ax, alpha=0.3)
    ax.plot([start_coords[0], end_coords[0]], [start_coords[1], end_coords[1]], 'k', linewidth=3)
    
#     grp.plot(ax=ax, color='yellow')
    ax.set(xlim=(xmin, xmax), ylim=(ymin, ymax), xlabel='X(m)', ylabel='Y(m)')
    ax.set(xlabel='X(m)', ylabel='Y(m)')
    plt.colorbar(im, ax=ax, label='Elevation (m)')
    #plt.savefig("DEM_2s")


In [ ]:
# Get the elevations along the 2D transect
meshsize_nx=100
gdf = get_xsec_elevation_3(start_coords, end_coords, f_dem, npts=meshsize_nx-1)
gdf

In [ ]:
# Get the centroil of the surface elements
lon_centroid = (gdf.lon[1:].values + gdf.lon[:-1].values) / 2
lat_centroid = (gdf.lat[1:].values + gdf.lat[:-1].values) / 2
surface_centroid = np.array([lon_centroid, lat_centroid]).T
surface_centroid.shape
#meshsize_nx=surface_centroid.shape[0]

In [ ]:
plt.rcParams['figure.figsize'] = [36, 5]
small_size = 15
medium_size = 25
bigger_size = 30
plt.rc('font', size=small_size)          # controls default text sizes
plt.rc('axes', titlesize=small_size)    # fontsize of the axes title
plt.rc('axes', labelsize=small_size)    # fontsize of the x and y labels
plt.rc('xtick', labelsize=small_size)    # fontsize of the tick labels
plt.rc('ytick', labelsize=small_size)    # fontsize of the tick labels
plt.rc('legend', fontsize=small_size)    # legend fontsize
plt.rc('figure', titlesize=small_size)  # fontsize of the figure title
plt.rc('text', usetex = False)
# Specify the top surface
x = (gdf[['h_distance']].values.flatten())
z = gdf[['elevation']].values.flatten()
print(f'# of x and z coordinates: {len(x)}, {len(z)}')

# plot the surface topography
plt.plot(x,z, 'k'); plt.xlabel('x distance (m)'); plt.ylabel('z elevation (m)')
#plt.xscale('log') 
#plt.savefig("Elevation_tempest_ext_t2",dpi=300)
plt.show()
# make the (manifold) 2D mesh.
# m2 = meshing_ats.Mesh2D.from_Transect(x,z)'''
m2 = watershed_workflow.mesh.Mesh2D.from_Transect(x,z)
#m2 = workflow.mesh.Mesh2D.from_Transect(x2,z2)
#m2 = workflow.mesh.Mesh2D.from_Transect(x3,z3)'''
print(f'# of cells: {m2.num_cells}')


## hydrologically condition the mesh

In [ ]:
dir(m2)

In [ ]:
# from watershed-workflow documentation, find two functions that would like to try
# 1 - watershed_workflow.condition.fill_pits_dual
# 2 - watershed_workflow.condition.identify_local_minima

local_minima = watershed_workflow.condition.identify_local_minima(m2)
# # plot the surface topography
# plt.plot(x,z, 'k'); plt.xlabel('x distance (m)'); plt.ylabel('z elevation (m)')

# for i in range(len(local_minima)):
#     if local_minima[i] == 1:
#         midpoint_x = (x[i] + x[i+1]) / 2
#         midpoint_z = (z[i] + z[i+1]) / 2
#         plt.plot(midpoint_x, midpoint_z, 'ro', markerfacecolor='none', markersize=10)  # Open red circle
# plt.show()

In [ ]:
# watershed_workflow.condition.fill_pits(m2, algorithm=3)
watershed_workflow.condition.fill_pits(m2, outlet=meshsize_nx, algorithm=3) # update 20250521

plt.figure(figsize=(14, 4))

# plot the surface topography
plt.plot(x,z, 'k', label='raw dem', linewidth=2); plt.xlabel('x distance (m)'); plt.ylabel('z elevation (m)')

for i in range(len(local_minima)):
    if local_minima[i] == 1:
        midpoint_x = (x[i] + x[i+1]) / 2
        midpoint_z = (z[i] + z[i+1]) / 2
        plt.plot(midpoint_x, midpoint_z, 'ro', markerfacecolor='none', markersize=10)  # Open red circle

plt.plot(m2.coords[:,0], m2.coords[0:,2], 'bo', label='conditioned', alpha=0.5, markersize=3)
plt.legend()
plt.show()

In [ ]:
boundary_nodes = m2.boundary_nodes
outlet = boundary_nodes[np.argmin(m2.coords[boundary_nodes, 2])]
print(m2.boundary_nodes)
print(outlet)
print(m2.coords[-1], m2.coords[-2])

In [ ]:
#print(m2.coords[40:44])

In [ ]:
print(m2.coords[0], m2.coords[-1])

In [ ]:
print(local_minima)
print(len(local_minima))
print(m2.coords[:,0])

# Surface properties

In [ ]:
# download the NLCD raster - HUGE, 30GB
# [Yi's comment, noticing that NLCD 2019 data was used. How about wildfire?]

xbox = box(*xsec.bounds, ccw=True)
target_bounds = gdf.total_bounds
crs = gdf.crs
lc_profile, lc_raster = watershed_workflow.get_raster_on_shape(sources['land cover'], xbox, crs)
lc = watershed_workflow.values_from_raster(surface_centroid, crs, lc_raster, lc_profile)

# what land cover types did we get?
logging.info('Found land cover dtypes: {}'.format(lc.dtype))
logging.info('Found land cover types: {}'.format(set(lc)))
lc


In [ ]:
### land cover histogram
u, count = np.unique(lc, return_counts=True)
count_sort_ind = np.argsort(-count)
lc_dict = dict(zip(u[count_sort_ind], count[count_sort_ind]))
print(lc_dict)
aa = count[count_sort_ind]
print(aa[0]/np.sum(aa)*100, aa[1]/np.sum(aa)*100, aa[2]/np.sum(aa)*100, aa[3]/np.sum(aa)*100)
plt.hist(lc, bins='auto')
plt.ylabel('counts')
plt.xlabel('index')
plt.show()

In [ ]:
# -- get the NLCD colormap which uses official NLCD colors and labels
nlcd_indices, nlcd_cmap, nlcd_norm, nlcd_ticks, nlcd_labels = \
                watershed_workflow.colors.generate_nlcd_colormap(lc)
nlcd_labels, nlcd_indices


In [ ]:
nlcd_labels_dict = dict(zip(nlcd_indices, nlcd_labels))
nlcd_labels_dict

In [ ]:
# we don't really need all of these.  Keep Evergreen, Deciduous, Shrub, and merge the rest into "Other"
nlcd_color_new = 99 * np.ones_like(lc)

groupings = {
    42 : ['Evergreen Forest',],
    41 : ['Deciduous Forest', 'Mixed Forest', 'Woody Wetlands'],
    52 : ['Dwarf Scrub', 'Shrub/Scrub', 'Grassland/Herbaceous', 'Sedge/Herbaceous', 
                     'Pasture/Hay', 'Cultivated Crops'],
    # 71 : ['Grassland/Herbaceous'],
}

for k,v in groupings.items():
    for label in v:
        index = sources['land cover'].indices[label]
        nlcd_color_new[np.where(lc == index)] = k
    
print(nlcd_color_new)


In [ ]:
### land cover histogram; after regroup
u, count = np.unique(nlcd_color_new, return_counts=True)
count_sort_ind = np.argsort(-count)
print(u[count_sort_ind])
print(count[count_sort_ind])
aa = count[count_sort_ind]
print(aa[0]/np.sum(aa)*100, aa[1]/np.sum(aa)*100, aa[2]/np.sum(aa)*100)#, aa[3]/np.sum(aa)*100)
plt.hist(nlcd_color_new, bins='auto')
plt.ylabel('counts')
plt.xlabel('index')
plt.show()

In [ ]:
nlcd_color_new_other_as_water = np.where(nlcd_color_new == 99, 11, nlcd_color_new)
                                         
# -- get the NLCD colormap which uses official NLCD colors and labels
nlcd_indices, nlcd_cmap, nlcd_norm, nlcd_ticks, nlcd_labels = \
                watershed_workflow.colors.generate_nlcd_colormap(nlcd_color_new_other_as_water)

# make (water, 11) into (other, 99)
nlcd_labels[0] = 'Other'
nlcd_indices[0] = 99
nlcd_labels, nlcd_indices


Add surface land cover type labeled sets to the mesh

Lastly, we can add these two land use types to the mesh, so that we can refer to them for LAI as a function of time in the ATS run.

In [ ]:
# add labeled sets to the mesh for NLCD; after regroup
nlcd_labels_dict = dict(zip(nlcd_indices, nlcd_labels))
watershed_workflow.regions.add_nlcd_labeled_sets(m2, nlcd_color_new, nlcd_labels_dict)
print(nlcd_labels_dict)

In [ ]:
# then we manually set the crosswalk:
nlcd_crosswalk_modis = {'Evergreen Forest' : 'NLCD Evergreen Forest',
                        'Shrub/Scrub'      : 'NLCD Shrub Scrub'}
print(nlcd_crosswalk_modis)

In [ ]:
# print out the regions formed
for ls in m2.labeled_sets:
    print(f'{ls.setid} : {ls.entity} : "{ls.name}", size = {len(ls.ent_ids)}')

# Subsurface properties

## Soil properties

In [ ]:
# download the NRCS soils data as shapes and project it onto the mesh

# -- download the shapes
#target_bounds = gdf.total_bounds
target_bounds = box(*gdf.total_bounds, ccw=True)

logging.info('target bounds: {}'.format(target_bounds))
soil_profile, soil_survey, soil_survey_props = watershed_workflow.get_shapes(sources['soil structure'], 
                                                                   target_bounds, crs, crs, properties=True)

# -- determine the NRCS mukey for each soil unit; this uniquely identifies soil 
#    properties
soil_ids = np.array([shp.properties['mukey'] for shp in soil_survey], np.int32)
#soil_survey_props.set_index('mukey', inplace=True)

# -- color a raster by the polygons (this makes identifying a triangle's value much 
#    more efficient)
# soil_color_raster, soil_color_profile, img_bounds = \
#             watershed_workflow.color_raster_from_shapes(target_bounds, 9, soil_survey,
#                                               soil_ids, crs)
soil_color_profile, soil_color_raster = \
            watershed_workflow.color_raster_from_shapes(soil_survey, crs, soil_ids,
                                              target_bounds.bounds, 9, crs, -1)

# -- resample the raster to the triangles
# soil_color = workflow.values_from_raster(m2.centroids(), crs, 
#                                          soil_color_raster, soil_color_profile)
soil_color = watershed_workflow.values_from_raster(surface_centroid, crs, 
                                         soil_color_raster, soil_color_profile)

# -- select only the soils within the watershed
soil_survey_props.set_index('mukey', inplace=True, drop=False)
soil_survey_props = soil_survey_props.loc[np.unique(soil_color), :]
soil_survey_props


In [ ]:
# what does soil thickness look like?
soil_thickness = np.empty(soil_color.shape, 'd')
for mukey in soil_survey_props.index:
    soil_thickness[soil_color == mukey] = soil_survey_props.loc[mukey,'thickness [cm]']

soil_thickness = soil_thickness / 100
soil_thickness

In [ ]:
# calcualte the median of soil thickness, used in `get_docflux_from_ELM_3D.ipynb` for processing DOC injection and [NH4+] [NO3] BC
soil_thickness_median = {'soil_thickness_median': np.median(soil_thickness)}
outputs['soil_thickness_median'] = f'../data-processed/{site_name}/soilmedianthickness_{site_name}.mat'
savemat(outputs['soil_thickness_median'], soil_thickness_median)

# loaded_data = loadmat(outputs['soil_thickness_median']) # verify
# print(loaded_data['soil_thickness_median'].item())

In [ ]:
# Note the missing data (white).  This is because some SSURGO map units have no formation with complete 
# information.  So we merge the above available data, filling where possible and dropping regions that
# do not have a complete set of properties.
soil_survey_props_clean = soil_survey_props.copy()

# later scripts expect 'native_index' as a standard name of holding onto the original IDs
soil_survey_props_clean.rename_axis('native_index', inplace=True)
soil_survey_props_clean.rename(columns={'mukey':'native_index'}, inplace=True)

# need thickness in m
soil_survey_props_clean['thickness [cm]'] = soil_survey_props_clean['thickness [cm]']/100.
soil_survey_props_clean.rename(columns={'thickness [cm]':'thickness [m]'}, inplace=True)

def replace_column_nans(df, col_nan, col_replacement):
    """In a df, replace col_nan entries by col_replacement if is nan.  In Place!"""
    row_indexer = df[col_nan].isna()
    df.loc[row_indexer, col_nan] = df.loc[row_indexer, col_replacement]
    return

# where poro or perm is nan, put Rosetta poro
replace_column_nans(soil_survey_props_clean, 'porosity [-]', 'Rosetta porosity [-]')
replace_column_nans(soil_survey_props_clean, 'permeability [m^2]', 'Rosetta permeability [m^2]')

# drop unnecessary columns
for col in ['Rosetta porosity [-]', 'Rosetta permeability [m^2]', 'bulk density [g/cm^3]', 'total sand pct [%]',
            'total silt pct [%]', 'total clay pct [%]']:
    soil_survey_props_clean.pop(col)
    
# drop nans
soil_survey_props_clean.dropna(inplace=True)
soil_survey_props_clean.reset_index(drop=True, inplace=True)
soil_survey_props_clean


In [ ]:
# create a new soil_color, keeping on those that are kept here and re-indexing to ATS indices
soil_color_new = -np.ones_like(soil_color)
for new_id, mukey in enumerate(soil_survey_props_clean['native_index']):
    soil_color_new[np.where(soil_color == mukey)] = 1000+new_id
soil_color_new


## Geological properties

In [ ]:
# extract the GLYHMPS geologic structure data as shapes and project it onto the mesh
# target_bounds = watershed.exterior().bounds
#target_bounds = gdf.total_bounds
target_bounds = box(*gdf.total_bounds, ccw=True)
logging.info('target bounds: {}'.format(target_bounds))

_, geo_survey, geo_survey_props = watershed_workflow.get_shapes(sources['geologic structure'], 
                                                      target_bounds.bounds, crs, crs, properties=True)

# -- log the bounds targeted and found
logging.info('shape union bounds: {}'.format(
    shapely.ops.cascaded_union(geo_survey).bounds))

# -- determine the ID for each soil unit; this uniquely identifies formation
#    properties
geo_ids = np.array([shp.properties['id'] for shp in geo_survey], np.int32)

# -- color a raster by the polygons (this makes identifying a triangle's value much 
#    more efficient)
# geo_color_raster, geo_color_profile, img_bounds = \
#             watershed_workflow.color_raster_from_shapes(target_bounds, 9, geo_survey,
#                                               geo_ids, crs)
geo_color_profile, geo_color_raster = \
            watershed_workflow.color_raster_from_shapes(geo_survey, crs, geo_ids,
                                              target_bounds.bounds, 9, crs, -1)

# -- resample the raster to the triangles
# geo_color = workflow.values_from_raster(m2.centroids(), crs, 
#                                          geo_color_raster, geo_color_profile)
geo_color = watershed_workflow.values_from_raster(surface_centroid, crs, 
                                         geo_color_raster, geo_color_profile)


# -- select the properties that appear in the mesh
geo_survey_props.set_index('id', inplace=True, drop=False)
geo_survey_props = geo_survey_props.loc[np.unique(geo_color), :]
geo_survey_props


In [ ]:
# note there are clearly some common regions -- no need to duplicate those with identical values.
geo_survey_props_clean = geo_survey_props.copy()
geo_survey_props_clean.pop('logk_stdev [-]')
geo_survey_props_clean.rename(columns={'id':'native_index'}, inplace=True)


def reindex_remove_duplicates(df, index=None):
    """Removes duplicates, creating a new index and saving the old index as tuples of duplicate values. In place!"""
    if index is not None:
        if index in df:
            df.set_index(index, drop=True, inplace=True)
    
    index_name = df.index.name

    # identify duplicate rows
    duplicates = list(df.groupby(list(df)).apply(lambda x: tuple(x.index)))
    # regarding the warning message, maybe
    # duplicates = list(df.groupby(list(df), include_groups=False).apply(lambda x: tuple(x.index)))

    # order is preserved
    df.drop_duplicates(inplace=True)
    df.reset_index(inplace=True)
    df[index_name] = duplicates
    return

reindex_remove_duplicates(geo_survey_props_clean, 'native_index')
geo_survey_props_clean


In [ ]:
# create a new geologic layer color, keeping on those that are kept here and re-indexing to ATS indices
geo_color_new = -np.ones_like(geo_color)
for new_id, old_id_dups in enumerate(geo_survey_props_clean['native_index']):
    for old_id in old_id_dups:
        geo_color_new[np.where(geo_color == old_id)] = 100+new_id
geo_color_new


## Depth-to-bedrock

In [ ]:
xbox = box(*xsec.bounds, ccw=True)
#DTB_source = workflow.source_list.structure_sources['SoilGrids2017']
#DTB_source = watershed_workflow.sources.manager_soilgrids_2017.FileManagerSoilGrids2017()
crs = gdf.crs
# DTB_profile, DTB_raster = watershed_workflow.get_raster_on_shape(DTB_source, xbox, crs, 
#                                                        nodata=-99999, variable='BDTICM')
DTB_profile, DTB_raster = watershed_workflow.get_raster_on_shape(sources['depth to bedrock'], xbox, crs, 
                                                       nodata=-99999)#, variable='BDTICM')
 
# resample the raster to the triangles
DTB_raster = DTB_raster/100 #convert from cm to m
#DTB = workflow.values_from_raster(m2.centroids(), crs, DTB_raster, DTB_profile)
DTB = watershed_workflow.values_from_raster(surface_centroid, crs, DTB_raster, DTB_profile)
DTB = np.where(DTB >= 0, DTB, np.nan)
DTB

# Mesh extrusion



In [ ]:
 # Given the surface mesh and material IDs on both the surface and subsurface, we can extrude the surface mesh in the vertical to make a 3D mesh.

#First, all integer IDs in Exodus files must be unique.  This includes Material IDs, side sets, etc.  We create the Material ID map and data frame.  This is used to standardize IDs from multiple data sources.  Traditionally, ATS numbers Material IDs/Side Sets as:

#* 0-9 : reserved for boundaries, surface/bottom, etc
#* 10-99 : Land Cover side sets, typically NLCD IDs are used
#* 100-999 : geologic layer material IDs. 999 is reserved for bedrock.
#* 1000-9999 : soil layer material IDs

In [ ]:
# map SSURGO mukey to ATS_ID
soil_survey_props_clean['ats_id'] = range(1000, 1000+len(soil_survey_props_clean))
soil_survey_props_clean.set_index('ats_id', inplace=True)

# map GLHYMPS id to ATS_ID
geo_survey_props_clean['ats_id'] = range(100, 100+len(geo_survey_props_clean))
geo_survey_props_clean.set_index('ats_id', inplace=True)

bedrock_props = watershed_workflow.soil_properties.get_bedrock_properties()

# merge the properties databases
subsurface_props = pd.concat([geo_survey_props_clean,
                                  soil_survey_props_clean,
                                  bedrock_props])

# save the properties to disk for use in generating input file
outputs['subsurface_properties_filename'] = f'../data-processed/{site_name}/{site_name}_subsurface_properties.csv'
subsurface_props.to_csv(outputs['subsurface_properties_filename'])
# subsurface_props.to_csv('watershed_subsurface_properties.csv')
subsurface_props


In [ ]:
   """ wrm['region'] = region
    wrm['van Genuchten alpha [Pa^-1]'] = float(props['van Genuchten alpha [Pa^-1]'])
    wrm['van Genuchten n [-]'] = float(props['van Genuchten n [-]'])
    wrm['residual saturation [-]'] = 0.05 #float(props['residual saturation [-]'])
    wrm['smoothing interval width [saturation]'] = 0.05 """

In [ ]:
#Next we extrude the DEM to create a 3D mesh.

#The most difficult aspect of extrusion is creating meshes that:
#1. aren't huge numbers of cells
#2. aren't huge cell thicknesses, especially near the surface
#3. follow implied interfaces, e.g. bottom of soil and bottom of geologic layer

#This is an iterative process that requires some care and some art.

In [ ]:
# here we choose the bottom of the domain to be the maximum of the depth to bedrock.  
# This is really up to the user, but we are hard-coding this for this watershed_workflow.
dtb_max = np.nanmax(DTB)
DTB = np.where(np.isnan(DTB), dtb_max, DTB)

total_thickness = np.ceil(DTB.max())
print(f'total thickness: {total_thickness} m')

# total_thickness = 41.0

In [ ]:
# Generate a dz structure for the top 2m of soil

# here we try for 10 cells, starting at 5cm at the top and going to 50cm at the bottom of the 2m thick soil
#dzs, res = workflow.mesh.optimize_dzs(0.05, 0.5, 2, 10) 20220227_BL change the mesh
dzs, res = watershed_workflow.mesh.optimize_dzs(0.01, 0.2, 2, 20)
print(dzs, sum(dzs))

# this looks like it would work out, with rounder numbers:
dzs_soil = [0.05, 0.05, 0.05, 0.1, 0.25, 0.5, 0.5, 0.5]
print(sum(dzs_soil), len(dzs_soil))

In [ ]:
# [Yi Xiao] try and setup the z mesh here. Starting from optimize_dzs, and then set your own dzs_geo
# total thickness, minus 2m soil thickness, leaves us with total_thickness - 2 meters to make up.
# optimize again...
dzs2, res2 = watershed_workflow.mesh.optimize_dzs(1, 3, total_thickness-2, 11)
print(dzs2)
print(sum(dzs2),len(dzs2))

# how about...
dzs_geo = [1., 1., 1.5, 1.5, 2., 2., 2., 3., 3., 3.]
print(dzs_geo)
print(sum(dzs_geo), len(dzs_geo))

In [ ]:
# layer extrusion
# -- data structures needed for extrusion
layer_types = []
layer_data = []
layer_ncells = []
layer_mat_ids = []

# -- soil layer --
depth = 0
for dz in dzs_soil:
    depth += 0.5 * dz
    layer_types.append('constant')
    layer_data.append(dz)
    layer_ncells.append(1)
    
    br_or_geo = np.where(depth < DTB, geo_color_new, 999)
    soil_or_br_or_geo = np.where(np.bitwise_and(soil_color_new > 0, depth < soil_thickness),
                                 soil_color_new,
                                 br_or_geo)
    
    layer_mat_ids.append(soil_or_br_or_geo)
    depth += 0.5 * dz
    
# -- geologic layer --
for dz in dzs_geo:
    depth += 0.5 * dz
    layer_types.append('constant')
    layer_data.append(dz)
    layer_ncells.append(1)
    layer_mat_ids.append(np.where(depth < DTB, geo_color_new, 999))
    depth += 0.5 * dz

# print the summary
watershed_workflow.mesh.Mesh3D.summarize_extrusion(layer_types, layer_data, layer_ncells, layer_mat_ids)

# downselect subsurface properties to only those that are used
layer_mat_id_used = list(np.unique(np.array(layer_mat_ids)))
subsurface_props_used = subsurface_props.loc[layer_mat_id_used]
len(subsurface_props_used)


In [ ]:
# make the mesh, save it as an exodus file
m3 = watershed_workflow.mesh.Mesh3D.extruded_Mesh2D(m2, layer_types, layer_data, 
                                             layer_ncells, layer_mat_ids)


In [ ]:
print(nlcd_indices)
print(nlcd_labels)

In [ ]:
print('2D labeled sets')
print('---------------')
for ls in m2.labeled_sets:
    print(f'{ls.setid} : {ls.entity} : {len(ls.ent_ids)} : "{ls.name}"')

print('')
print('Extruded 3D labeled sets')
print('------------------------')
for ls in m3.labeled_sets:
    print(f'{ls.setid} : {ls.entity} : {len(ls.ent_ids)} : "{ls.name}"')

print('')
print('Extruded 3D side sets')
print('---------------------')
for ls in m3.side_sets:
    print(f'{ls.setid} : FACE : {len(ls.cell_list)} : "{ls.name}"')

In [ ]:
# [to-do] a simple merge of notebook from Zhi and Bing. I don't think both converted exo and vtk are needed.

# Save
mesh_filename_site = f'../data-processed/{site_name}/2dtransect_{site_name}_nx{meshsize_nx}_nz{len(dzs_soil)+len(dzs_geo)}.exo'
outputs['mesh_filename_site'] = mesh_filename_site

mesh_format_filename_site = f'../data-processed/{site_name}/2dtransect_{site_name}_nx{meshsize_nx}_nz{len(dzs_soil)+len(dzs_geo)}_format.exo'

try:
    os.remove(outputs['mesh_filename_site'])
    os.remove(mesh_format_filename_site)
except FileNotFoundError:
    pass
m3.write_exodus(outputs['mesh_filename_site'])

# Now convert the file from "polyhedral" to "fixed format" and open it in VisIt or Paraview.
os.system("$AMANZI_TPLS_DIR/bin/meshconvert {} {}".format(outputs['mesh_filename_site'], mesh_format_filename_site))

# exo file can be open with xarray
# import xarray as xr
# mesh_format_filename = f'../data-processed/{site_name}/2dtransect_{site_name}_nx{meshsize_nx}_nz{len(dzs_soil)+len(dzs_geo)}_format.exo'
# mesh_format_loaded = xr.open_dataset(mesh_format_filename)
# mesh_format_loaded

In [ ]:
# # for add regions
# left, right   = float(x[0]), float(x[-1])
# front, back   = -0.5, 0.5
# topl, bottoml = float(z[0]), float(z[0])-total_thickness
# topr, bottomr = float(z[-1]), float(z[-1])-total_thickness
# left, right, topl, bottoml, topr, bottomr

# [update v250521]
# - z is before conditioned, should use m2.coords[:,0] and m2.coords[:,:]
# - save m2 to m2_{site_name}_nx{meshsize_nx}.mat for later use

# for add regions
left, right   = float(m2.coords[0,0]), float(m2.coords[-1,0])
front, back   = -0.5, 0.5
topl, bottoml = float(m2.coords[0,2]), float(m2.coords[0,2])-total_thickness
topr, bottomr = float(m2.coords[-1,2]), float(m2.coords[-1,2])-total_thickness
left, right, topl, bottoml, topr, bottomr

# Convert GeoDataFrame gdf to numpy arrays
gdf_dict = {
    'lon': gdf['lon'].to_numpy(),
    'lat': gdf['lat'].to_numpy(),
    'h_distance': gdf['h_distance'].to_numpy(),
    'elevation': gdf['elevation'].to_numpy()
}

data_to_save = {
    'meshsize_nx': meshsize_nx,
    'dzs_soil': np.array(dzs_soil),
    'dzs_geo': np.array(dzs_geo),
    'm2_coords': m2.coords,
    'gdf_data': gdf_dict
}

m2_mat_filename =  f'../data-processed/{site_name}/m2_{site_name}_nx{meshsize_nx}.mat'
savemat(m2_mat_filename, data_to_save)

# get DayMet for hillslope

- **run notebook `get_Daymet.ipynb` before move on**
- Input
    - `data-processed/{site_name}/m2_{site_name}_nx100.mat`
- Output
    - `data-processed/{watershed_name}/{watershed_name}_DayMet_2013_2023.h5`
    - `data-processed/{site_name}/{site_name}_DayMet_2013_2023.h5`
    - `data-processed/{site_name}/{site_name}_DayMet_typical10yr_2013_2023.h5`

In [ ]:
user_completed_getDaymet = False #False  # User should change this to True

assert user_completed_getDaymet, (
    "Please complete get_Daymet.ipynb and set user_completed_getDaymet = True "
    "before running the remaining cells."
)

In [ ]:
# double check these configurations are the same in `get_Daymet.ipynb`
# use data from start_year_spinup to end_year_spinup, to generate typical year data for 1) steady-state spinup and 2) cyclic spinup. 
# 1) cyclic for nyears_steadystate_spinup years for steady-state spinup
# 2) cyclic for nyears_cyclic_spinup years for cyclic spinup
# start_year_spinup         = 2011 # in config.json
# end_year_spinup           = 2015
# nyears_steadystate_spinup = 10
# nyears_cyclic_spinup      = 10
# start_year_transient      = 2016
# end_year_transient        = 2020

outputs['daymet_filename_watershed'] = f'../data-processed/{watershed_name}/{watershed_name}_DayMet_{start_year_spinup}_{end_year_transient}.h5'
outputs['daymet_transient_filename_site'] = f'../data-processed/{site_name}/{site_name}_DayMet_{start_year_transient}_{end_year_transient}.h5'
outputs['daymet_spinup_filename_site'] = f'../data-processed/{site_name}/{site_name}_DayMet_typical{nyears_cyclic_spinup}yr_{start_year_spinup}_{end_year_spinup}.h5'

In [ ]:
# load h5 file; data structure is groups with time slices
with h5.File(outputs['daymet_transient_filename_site'], 'r') as hf:
    time_data = hf['time [s]'][:]
    num_timesteps = len(time_data)

    prain = np.array([hf['precipitation rain [m s^-1]'][str(i)][:] for i in range(num_timesteps)])
    psnow = np.array([hf['precipitation snow [m SWE s^-1]'][str(i)][:] for i in range(num_timesteps)])
    qswin = np.array([hf['incoming shortwave radiation [W m^-2]'][str(i)][:] for i in range(num_timesteps)])
    airtemp = np.array([hf['air temperature [K]'][str(i)][:] for i in range(num_timesteps)])
    vapor_pressure = np.array([hf['vapor pressure air [Pa]'][str(i)][:] for i in range(num_timesteps)])

In [ ]:
# plot the transient DayMet h5 file for ATS

#times = np.array([datetime.datetime(*t.timetuple()[0:6]) for t in raw_dat_2dtran_warped_ats.times])
times = [datetime.datetime(start_year_transient, 1, 1) + datetime.timedelta(seconds=int(t)) for t in time_data] # just use 2000 for spinup

prain_spatial_mean = np.mean(prain, axis=(1, 2))
psnow_spatial_mean = np.mean(psnow, axis=(1, 2))
qswin_spatial_mean = np.mean(qswin, axis=(1, 2))
airtemp_spatial_mean = np.mean(airtemp, axis=(1, 2))
vapor_pressure_spatial_mean = np.mean(vapor_pressure, axis=(1, 2))

fig = plt.figure(figsize=(15,6))
ax = fig.add_subplot(221)
ax.plot(times, prain_spatial_mean, 'b', label='rain')
ax.plot(times, psnow_spatial_mean, 'c', label='snow')
ax.set_ylabel('precip [m s^-1]')
ax.legend()

ax = fig.add_subplot(222)
ax.plot(times, qswin_spatial_mean, 'r')
ax.set_ylabel('incoming shortwave radiation [W m^-2]')

ax = fig.add_subplot(223)
ax.plot(times, airtemp_spatial_mean, 'g')
ax.set_ylabel('air temperature [K]')

ax = fig.add_subplot(224)
ax.plot(times, vapor_pressure_spatial_mean, 'm')
ax.set_ylabel('vapor pressure air [Pa]')
plt.show()

In [ ]:
# load h5 file
with h5.File(outputs['daymet_spinup_filename_site'], 'r') as hf:
    time_data = hf['time [s]'][:]
    num_timesteps = len(time_data)

    prain = np.array([hf['precipitation rain [m s^-1]'][str(i)][:] for i in range(num_timesteps)])
    psnow = np.array([hf['precipitation snow [m SWE s^-1]'][str(i)][:] for i in range(num_timesteps)])
    qswin = np.array([hf['incoming shortwave radiation [W m^-2]'][str(i)][:] for i in range(num_timesteps)])

In [ ]:
# plot again, precip typical year

#times = np.array([datetime.datetime(*t.timetuple()[0:6]) for t in dat_2dtran_smooth_ats.times])
times = [datetime.datetime(2000, 1, 1) + datetime.timedelta(seconds=int(t)) for t in time_data] # just use 2000 for spinup
prain_spatial_mean = np.mean(prain, axis=(1, 2))
psnow_spatial_mean = np.mean(psnow, axis=(1, 2))
qswin_spatial_mean = np.mean(qswin, axis=(1, 2))

fig = plt.figure(figsize=(15,3))
ax = fig.add_subplot(121)
ax.plot(times, prain_spatial_mean, 'b', label='rain')
ax.plot(times, psnow_spatial_mean, 'c', label='snow')

ax.set_ylabel('precip [m s^-1]')
ax.legend()

ax = fig.add_subplot(122)
ax.plot(times, qswin_spatial_mean, 'r')
ax.set_ylabel('incoming shortwave radiation [W m^-2]')
plt.show()

In [ ]:
rain = 'precipitation rain [m s^-1]'
snow = 'precipitation snow [m SWE s^-1]'
logging.info(f"try reading file {outputs['daymet_spinup_filename_site']}")

with h5.File(outputs['daymet_spinup_filename_site'], 'r') as fid:
    mean_precip4atsrun0 = np.array([fid[rain][ts][:].mean() + fid[snow][ts][:].mean() for ts in fid[rain].keys()]).mean()

logging.info(f'Mean annual precip rate [m s^-1] = {mean_precip4atsrun0}')

# get LAI-time for hillslope

**run `get_MODIS-LAI_OakCreek.ipynb` before move on**

In [ ]:
user_completed_getLAI = False #False  # User should change this to True

assert user_completed_getLAI, (
    "Please complete get_MODIS-LAI_OakCreek.ipynb and set user_completed_getLAI = True "
    "before running the remaining cells."
)

In [ ]:
# double check these configurations are the same in `get_MODIS-LAI_OakCreek.ipynb`
# use data from start_year_spinup to end_year_spinup, to generate typical year data for 1) steady-state spinup and 2) cyclic spinup. 
# 1) cyclic for nyears_steadystate_spinup years for steady-state spinup
# 2) cyclic for nyears_cyclic_spinup years for cyclic spinup
# start_year_spinup         = 2011 # in config.json
# end_year_spinup           = 2015
# nyears_steadystate_spinup = 10
# nyears_cyclic_spinup      = 10
# start_year_transient      = 2016
# end_year_transient        = 2020

startdate_transient  = f"{start_year_transient}-1-1"
enddate_transient    = f"{end_year_transient+1}-1-1"

outputs['modis_filename_watershed_raw']                = f'../data-processed/{watershed_name}/{watershed_name}_MODIS_LAI_20020704_20250124.h5'
outputs['modis_transient_filename_watershed_smoothed'] = f'../data-processed/{watershed_name}/{watershed_name}_MODIS_LAI_{startdate_transient}_{enddate_transient}_smoothed.h5'
outputs['modis_spinup_filename_watershed_smoothed']    = f'../data-processed/{watershed_name}/{watershed_name}_MODIS_LAI_typical{nyears_cyclic_spinup}yr_{start_year_spinup}_{end_year_spinup}.h5'

In [ ]:
# load h5 file; data structure is groups with time slices
with h5.File(outputs['modis_transient_filename_watershed_smoothed'], 'r') as d:
    modislai_smoothed = pd.DataFrame()
    for k in d.keys():
        modislai_smoothed[k] = d[k][:]

display(modislai_smoothed)

## plot: find columns with names starting with "NLCD"
nlcd_columns = [col for col in modislai_smoothed.columns if col.startswith("NLCD")]
modislai_smoothed[nlcd_columns].plot(figsize=(8, 4))

In [ ]:
# load h5 file; data structure is groups with time slices
with h5.File(outputs['modis_spinup_filename_watershed_smoothed'], 'r') as d:
    modislai_typical = pd.DataFrame()
    for k in d.keys():
        modislai_typical[k] = d[k][:]

display(modislai_typical)

## plot: find columns with names starting with "NLCD"
nlcd_columns = [col for col in modislai_typical.columns if col.startswith("NLCD")]
modislai_typical[nlcd_columns].plot(figsize=(8, 4))

# get Boundary Head for hillslope

**run `get_BChead.ipynb` before move on**

In [ ]:
user_completed_getBChead = False #False  # User should change this to True

assert user_completed_getBChead, (
    "Please complete get_BChead.ipynb and set user_completed_getBChead = True "
    "before running the remaining cells."
)

In [ ]:
outputs['BChead_start_spinup_filename_site'] = f'../data-processed/{site_name}/startpt_head_{nyears_cyclic_spinup}y_typical.h5'
outputs['BChead_end_spinup_filename_site'] = f'../data-processed/{site_name}/endpt_head_{nyears_cyclic_spinup}y_typical.h5'

startdate_transient       = f"{start_year_transient}-01-01"
enddate_transient         = f"{end_year_transient}-12-31"
outputs['BChead_start_transient_filename_site'] = f'../data-processed/{site_name}/startpt_head_{startdate_transient}_{enddate_transient}.h5'
outputs['BChead_end_transient_filename_site'] = f'../data-processed/{site_name}/endpt_head_{startdate_transient}_{enddate_transient}.h5'

In [ ]:
# load h5 file; data structure is groups with time slices
with h5.File(outputs['BChead_start_transient_filename_site'], 'r') as d:
    startpt_head = d['startpt_head'][:]
    times_in_day = d["Time"][:]/86400

with h5.File(outputs['BChead_end_transient_filename_site'], 'r') as d:
    endpt_head = d['endpt_head'][:]

plt.figure(figsize=(10, 5))
plt.plot(times_in_day, startpt_head, label="Start Point Head", linestyle="-", color="blue")
plt.plot(times_in_day, endpt_head, label="End Point Head", linestyle="--", color="red")
# Formatting the plot
plt.xlabel("Time [days]")
plt.ylabel("Head Value")
plt.title("Start and End Point Head over Time")
plt.legend()
plt.show()

In [ ]:
# load h5 file; data structure is groups with time slices
with h5.File(outputs['BChead_start_spinup_filename_site'], 'r') as d:
    startpt_head_spinup = d['startpt_head'][:]
    time_spinup_in_day = d["Time"][:]/86400

with h5.File(outputs['BChead_end_spinup_filename_site'], 'r') as d:
    endpt_head_spinup = d['endpt_head'][:]

plt.figure(figsize=(10, 5))
plt.plot(time_spinup_in_day, startpt_head_spinup, label="Start Point Head", linestyle="-", color="blue")
plt.plot(time_spinup_in_day, endpt_head_spinup, label="End Point Head", linestyle="--", color="red")
# Formatting the plot
plt.xlabel("Time [days]")
plt.ylabel("Head Value")
plt.title("Start and End Point Head over Time - typical year")
plt.legend()
plt.show()

In [ ]:
logging.info(f"Loaded startpt data with shape: {startpt_head.shape}")
logging.info(f"Loaded endpt data with shape: {endpt_head.shape}")

# Calculate the mean values for steady-state spinup
mean_bchead_startpt = np.mean(startpt_head_spinup)
mean_bchead_endpt = np.mean(endpt_head_spinup)

logging.info(f'Mean boundary head [m] for uphill, i.e. start point = {mean_bchead_startpt}')
logging.info(f'Mean boundary head [m] for outlet, i.e. end point = {mean_bchead_endpt}')

# Write ATS input deck

- to do, modify template or code to make this automatic for our hillslope simulations

In [ ]:
# prepare folders
os.makedirs(f'../caseflow-run0/{site_name}', exist_ok=True)
os.makedirs(f'../caseflow-run1/{site_name}', exist_ok=True)
os.makedirs(f'../caseflow-run2/{site_name}', exist_ok=True)

In [ ]:
# for add regions - left_face/right_face/front_face/back_face/surface left/surface right
def add_region_plane(main_list, name, point, normal):
    ats_input_spec.public.add_region(main_list,
                                    region_name=name,
                                    region_type='plane',
                                    region_args={'point' : point, 
                                                 'normal' : normal})

def add_region_box(main_list, name, low_coord, high_coord):
    ats_input_spec.public.add_region(main_list,
                                    region_name=name,
                                    region_type='box',
                                    region_args={'low coordinate' : low_coord, 
                                                 'high coordinate' : high_coord})

In [ ]:
# add the subsurface and surface domains
#
# Note this also adds a "computational domain" region to the region list, and a vis spec 
# for "domain"
def add_domains(main_list, mesh_filename, surface_region='surface', snow=True, canopy=True):
    ats_input_spec.public.add_domain(main_list, 
                                 domain_name='domain', 
                                 dimension=3, 
                                 mesh_type='read mesh file',
                                 mesh_args={'file':mesh_filename})
    if surface_region:
        main_list['mesh']['domain']['build columns from set'] = surface_region    
    
        # Note this also adds a "surface domain" region to the region list and a vis spec for 
        # "surface"
        ats_input_spec.public.add_domain(main_list,
                                domain_name='surface',
                                dimension=2,
                                mesh_type='surface',
                                mesh_args={'surface sideset name':'surface'})
        ats_input_spec.public.add_region(main_list,
                                region_name='surface boundary',
                                region_type='boundary',
                                region_args={'entity': 'FACE'})
        #add_region_plane(main_list, 'surface boundary demo', [right, 0], [1.0, 0.0])
        
        
    if snow:
        # Add the snow and canopy domains, which are aliases to the surface
        ats_input_spec.public.add_domain(main_list,
                                domain_name='snow',
                                dimension=2,
                                mesh_type='aliased',
                                mesh_args={'target':'surface'})
    if canopy:
        ats_input_spec.public.add_domain(main_list,
                                domain_name='canopy',
                                dimension=2,
                                mesh_type='aliased',
                                mesh_args={'target':'surface'})

    # from Bing's notebook, to add regions - left_face/right_face/front_face/back_face/surface left/surface right
    # Add the surrounding faces
    add_region_plane(main_list, 'left_face',  [left, front, 0.0], [-1.0, 0.0, 0.0])
    add_region_plane(main_list, 'right_face', [right, back, 0.0], [1.0, 0.0, 0.0])
    add_region_plane(main_list, 'front_face', [left, front, 0.0], [0.0, -1.0, 0.0])
    add_region_plane(main_list, 'back_face',  [right, back, 0.0], [0.0, 1.0, 0.0])
    
    # Add the left and right surface face
    add_region_box(main_list, 'surface left', [left, front], [left, back])
    add_region_box(main_list, 'surface right', [right, front], [right, back])
    
    #print(main_list["regions"])

In [ ]:
def add_land_cover(main_list):
    # next write a land-cover section for each NLCD type
    for index, nlcd_name in zip(nlcd_indices, nlcd_labels):
        ats_input_spec.public.set_land_cover_default_constants(main_list, nlcd_name)

    land_cover_list = main_list['state']['initial conditions']['land cover types']
    # update some defaults
    # ['Other', 'Deciduous Forest', 'Evergreen Forest', 'Shrub/Scrub']
    # note, these are from the CLM Technical Note v4.5
    #
    # Rooting depth curves from CLM TN 4.5 table 8.3
    #
    # Note, the mafic potential values are likely pretty bad for the types of van Genuchten 
    # curves we are using (ETC -- add paper citation about this topic).  Likely they need
    # to be modified.  Note that these values are in [mm] from CLM TN 4.5 table 8.1, so the 
    # factor of 10 converts to [Pa]
    #
    # Note, albedo of canopy taken from CLM TN 4.5 table 3.1
    if 42 in nlcd_color_new:
        land_cover_list['Evergreen Forest']['rooting profile alpha [-]'] = 7.0
        land_cover_list['Evergreen Forest']['rooting profile beta [-]'] = 2.0
        land_cover_list['Evergreen Forest']['rooting depth max [m]'] = 10.0
        land_cover_list['Evergreen Forest']['capillary pressure at fully closed stomata [Pa]'] = 255000
        land_cover_list['Evergreen Forest']['capillary pressure at fully open stomata [Pa]'] = 66000 * .1
        land_cover_list['Evergreen Forest']['albedo of canopy [-]'] = 0.07

    if 41 in nlcd_color_new:
        land_cover_list['Deciduous Forest']['rooting profile alpha [-]'] = 6.0
        land_cover_list['Deciduous Forest']['rooting profile beta [-]'] = 2.0
        land_cover_list['Deciduous Forest']['rooting depth max [m]'] = 10.0
        land_cover_list['Deciduous Forest']['capillary pressure at fully closed stomata [Pa]'] = 224000
        land_cover_list['Deciduous Forest']['capillary pressure at fully open stomata [Pa]'] = 35000 * .10
        land_cover_list['Deciduous Forest']['albedo of canopy [-]'] = 0.1

In [ ]:
# add soil sets: note we need a way to name the set, so we use, e.g. SSURGO-MUKEY.
def soil_set_name(ats_id):
    if ats_id == 999:
        return 'bedrock'
    source = subsurface_props_used.loc[ats_id]['source']
    native_id = subsurface_props_used.loc[ats_id]['native_index']
    if type(native_id) in [tuple,list]:
        native_id = native_id[0]
    return f"{source}-{native_id}"


# get an ATS "main" input spec list -- note, this is a dummy and is not used to write any files yet
def get_main():
    main_list = ats_input_spec.public.get_main()
    
    # flow_pk = ats_input_spec.public.add_leaf_pk(main_list, 'flow', main_list['cycle driver']['PK tree'], 
    #                                         'richards-spec')

    # add the mesh and all domains
    mesh_filename_site = os.path.join('..', outputs['mesh_filename_site'])
    add_domains(main_list, mesh_filename_site)

    # add labeled sets
    for ls in m3.labeled_sets:
        ats_input_spec.public.add_region_labeled_set(main_list, ls.name, ls.setid, mesh_filename_site, ls.entity)
    for ss in m3.side_sets:
        ats_input_spec.public.add_region_labeled_set(main_list, ss.name, ss.setid, mesh_filename_site, 'FACE')
    
    # add land cover
    add_land_cover(main_list)

    # add soil material ID regions, porosity, permeability, and WRMs
    for ats_id in subsurface_props_used.index:
        props = subsurface_props_used.loc[ats_id]
        set_name = soil_set_name(ats_id)
        
        if props['van Genuchten n [-]'] < 1.5:
            smoothing_interval = 0.01
        else:
            smoothing_interval = 0.0
        
        ats_input_spec.public.add_soil_type(main_list, set_name, ats_id, mesh_filename_site,
                                            float(props['porosity [-]']),
                                            float(props['permeability [m^2]']), 1.e-7,
                                            float(props['van Genuchten alpha [Pa^-1]']),
                                            float(props['van Genuchten n [-]']),
                                            float(props['residual saturation [-]']),
                                            float(smoothing_interval))    
    
    return main_list

# # create the main list
# main_list = get_main()

# outputs['generated_ats'] = f'../data-processed/{name}/{name}_generated_ats.xml'
# ats_input_spec.io.write(main_list, outputs['generated_ats'])
# main_xml = ats_input_spec.io.to_xml(main_list)

In [ ]:
# [to do] homogeneous_wrm, homogeneous_poro, homogeneous_perm are not functioning
# see populate_basic_properties in Zhi's old notebook
def populate_basic_properties(xml, main_xml, homogeneous_wrm=False, homogeneous_poro=False, homogeneous_perm=False):
    """This function updates an xml object with the above properties for mesh, regions, soil props, and lc props"""
    # based on coweeta example from watershed-workflow v1.5
    # find and replace the mesh list
    xml.replace('mesh', asearch.child_by_name(main_xml, 'mesh'))

    # find and replace the regions list
    xml.replace('regions', asearch.child_by_name(main_xml, 'regions'))
    
    # update all model parameters lists
    xml_parlist = asearch.find_path(xml, ['state', 'model parameters'], no_skip=True)
    for parlist in asearch.find_path(main_xml, ['state', 'model parameters'], no_skip=True):
        try:
            xml_parlist.replace(parlist.getName(), parlist)
        except aerrors.MissingXMLError:
            xml_parlist.append(parlist)

    # update all evaluator lists
    xml_elist = asearch.find_path(xml, ['state', 'evaluators'], no_skip=True)
    for elist in asearch.find_path(main_xml, ['state', 'evaluators'], no_skip=True):
        try:
            xml_elist.replace(elist.getName(), elist)
        except aerrors.MissingXMLError:
            xml_elist.append(elist)    
    
    # find and replace land cover
    consts_list = asearch.find_path(xml, ['state', 'initial conditions'])
    lc_list = asearch.find_path(main_xml, ['state', 'initial conditions', 'land cover types'], no_skip=True)
    try:
        consts_list.replace('land cover types', lc_list)
    except aerrors.MissingXMLError:
        consts_list.append(lc_list)

    # # update all [observations][fluxes] lists
    # xml_elist = asearch.find_path(xml, ['observations', 'fluxes'], no_skip=True)
    # for elist in asearch.find_path(main_xml, ['observations', 'fluxes'], no_skip=True):
    #     try:
    #         xml_elist.replace(elist.getName(), elist)
    #     except aerrors.MissingXMLError:
    #         xml_elist.append(elist)    

# def create_unique_name(name, homogeneous_wrm=False, homogeneous_poro=False, homogeneous_perm=False):
#     suffix = '_h'
#     if homogeneous_perm:
#         suffix += 'K'
#     if homogeneous_poro:
#         suffix += 'p'
#     if homogeneous_wrm:
#         suffix += 'w'
#     if suffix == '_h':
#         suffix = ''
#     return name + suffix

## write xml - r0 spinup_steadystate

In [ ]:
def hillslope_head_bc_constant(startpt_bcname, startpt_region, startpt_head, endpt_bcname, endpt_region, endpt_head):
    pl = ParameterList(name="head")
    pl1 = ParameterList(name=startpt_bcname)
    pl1a = Parameter(name="regions", ptype="Array(string)", value=startpt_region)
    pl1b = ParameterList(name="boundary head")
    pl1b1 = ParameterList(name="function-constant")
    pl1b1a = Parameter(name="value", ptype="double", value=startpt_head)
    
    pl2 = ParameterList(name=endpt_bcname)
    pl2a = Parameter(name="regions", ptype="Array(string)", value=endpt_region)
    pl2b = ParameterList(name="boundary head")
    pl2b1 = ParameterList(name="function-constant")
    pl2b1a = Parameter(name="value", ptype="double", value=endpt_head)
    
    pl1b1.append(pl1b1a)
    pl1b.append(pl1b1)
    pl1.append(pl1a); pl1.append(pl1b)

    pl2b1.append(pl2b1a)
    pl2b.append(pl2b1)
    pl2.append(pl2a); pl2.append(pl2b)
    
    pl.append(pl1)
    pl.append(pl2)
    return pl

def hillslope_head_bc_h5file(startpt_bcname, startpt_region, startpt_file, startpt_xheader, startpt_yheader, endpt_bcname, endpt_region, endpt_file, endpt_xheader, endpt_yheader):
    pl = ParameterList(name="head")
    
    # Start point boundary condition
    pl1 = ParameterList(name=startpt_bcname)
    pl1a = Parameter(name="regions", ptype="Array(string)", value=startpt_region)
    pl1b = ParameterList(name="boundary head")
    pl1b1 = ParameterList(name="function-tabular")
    pl1b1a = Parameter(name="file", ptype="string", value=startpt_file)
    pl1b1b = Parameter(name="x header", ptype="string", value=startpt_xheader)
    pl1b1c = Parameter(name="y header", ptype="string", value=startpt_yheader)
    pl1b1d = Parameter(name="form", ptype="Array(string)", value="{linear}")
    
    # Build start point structure
    pl1b1.append(pl1b1a)
    pl1b1.append(pl1b1b)
    pl1b1.append(pl1b1c)
    pl1b1.append(pl1b1d)
    pl1b.append(pl1b1)
    pl1.append(pl1a)
    pl1.append(pl1b)
    
    # End point boundary condition
    pl2 = ParameterList(name=endpt_bcname)
    pl2a = Parameter(name="regions", ptype="Array(string)", value=endpt_region)
    pl2b = ParameterList(name="boundary head")
    pl2b1 = ParameterList(name="function-tabular")
    pl2b1a = Parameter(name="file", ptype="string", value=endpt_file)
    pl2b1b = Parameter(name="x header", ptype="string", value=endpt_xheader)
    pl2b1c = Parameter(name="y header", ptype="string", value=endpt_yheader)
    pl2b1d = Parameter(name="form", ptype="Array(string)", value="{linear}")
    
    # Build end point structure
    pl2b1.append(pl2b1a)
    pl2b1.append(pl2b1b)
    pl2b1.append(pl2b1c)
    pl2b1.append(pl2b1d)
    pl2b.append(pl2b1)
    pl2.append(pl2a)
    pl2.append(pl2b)
    
    # Append start and end point configurations to 'head' parameter list
    pl.append(pl1)
    pl.append(pl2)
    
    return pl

In [ ]:
def write_spinup_steadystate(name, mean_precip, **kwargs):
    # name = create_unique_name(name, **kwargs)
    # logging.info(f'Writing transient: {name}')

    # create the main list
    main = get_main()

    # set precip to 0.6 * the mean precip value
    precip = main['state']['evaluators'].append_empty('surface-precipitation')
    precip.set_type('independent variable constant', ats_input_spec.public.known_specs['independent-variable-constant-evaluator-spec'])
    precip['value'] = float(mean_precip * 0.6)
    #print(main['state']['evaluators'])

    # Simulation time
    main['cycle driver']['start time'] = 0.0
    main['cycle driver']['start time units'] = 'd'
    main['cycle driver']['end time'] = 365.0*10
    main['cycle driver']['end time units'] = 'd'
    #print(main['cycle driver'])
    
    # write the spinup xml file
    # load the template file
    xml = aio.fromFile('caseflow-steadystate-template.ats1.5.xml') #template v1.5

    # UPDATE_P1: the template xml with the main xml generated here
    main_xml = ats_input_spec.io.to_xml(main)
    populate_basic_properties(xml, main_xml, **kwargs)

    # UPDATE_P2: runX specific configuration
    # P2.1: find and replace the [cycle driver][start time/start time units/end time/end time units]
    xml_stlist = asearch.find_path(xml, ['cycle driver'], no_skip=True)
    for tmp_name in ['start time', 'start time units', 'end time', 'end time units']:
        stlist = asearch.find_path(main_xml, ['cycle driver', tmp_name], no_skip=True)
        try:
            xml_stlist.replace(stlist.getName(), stlist)
        except aerrors.MissingXMLError:
            xml_parlist.append(stlist)

    # P2.2: find and replace the [PKs][subsurface flow][boundary conditions]
    flow_bc = asearch.find_path(xml, ['PKs', 'subsurface flow', 'boundary conditions'])
    flow_bc.append(hillslope_head_bc_constant("uphill", "left_face", mean_bchead_startpt, "outlet", "right_face", mean_bchead_endpt))
    print(flow_bc)
    
    # P2.3: find and replace the [PKs][surface flow][boundary conditions]
    landflow_bc = asearch.find_path(xml, ['PKs', 'surface flow', 'boundary conditions'])
    landflow_bc.append(hillslope_head_bc_constant("uphill", "surface left", mean_bchead_startpt, "outlet", "surface right", mean_bchead_endpt))
    print(landflow_bc)

    # write to disk
    outputs[f'spinup_steadystate_{name}_filename'] = f'../caseflow-run0/{name}_nx{meshsize_nx}_nz{len(dzs_soil)+len(dzs_geo)}.run0.v1.5.xml'
    aio.toFile(xml, outputs[f'spinup_steadystate_{name}_filename'])

    # the run directory
    outputs[f'spinup_steadystate_{name}_rundir'] = f'../caseflow-run0/{name}'


In [ ]:
write_spinup_steadystate(site_name, mean_precip4atsrun0, homogeneous_wrm=True, homogeneous_poro=True, homogeneous_perm=True)
# actually homogeneous_wrm, homogeneous_poro, and homogeneous_perm are not functioning currently
# write_spinup_steadystate(site_name, mean_precip) # equavalent

## write xml - r1 and r2 transient

In [ ]:
print(f'config.json start_year_spinup = {start_year_spinup}')
print(f'config.json end_year_spinup = {end_year_spinup}')
print(f'config.json nyears_cyclic_spinup = {nyears_cyclic_spinup}')
print(f'config.json start_year_transient = {start_year_transient}')
print(f'config.json end_year_transient = {end_year_transient}')

In [ ]:
def write_transient(name, cyclic_steadystate=False, start_year=start_year_spinup, end_year=end_year_spinup, **kwargs):
    # make a unique name based on options
    #name = create_unique_name(name, **kwargs)
    #logging.info(f'Writing transient: {name}')

    # create the main list
    main = get_main()

    # update the DayMet filenames
    if cyclic_steadystate:
        daymet_filename = outputs['daymet_spinup_filename_site']
    else:
        daymet_filename = outputs['daymet_transient_filename_site']
    ats_input_spec.public.add_daymet_box_evaluators(main, os.path.join('..', daymet_filename), True)

    # update the LAI filenames
    if cyclic_steadystate:
        lai_filename = outputs['modis_spinup_filename_watershed_smoothed']
    else:
        lai_filename = outputs['modis_transient_filename_watershed_smoothed']
    ats_input_spec.public.add_lai_point_evaluators(main, os.path.join('..', lai_filename), 
                                                   list(nlcd_labels_dict.values()), nlcd_crosswalk_modis)

    if cyclic_steadystate:
        prefix = 'cyclic_steadystate'
        previous = 'spinup_steadystate'
        runnum = 'run1'
    else:
        prefix = 'transient'
        previous = 'cyclic_steadystate'
        runnum = 'run2'

    template_filename = f'caseflow-{prefix}-template.ats1.5.xml'
    
    # write the cyclic spinup xml file
    # load the template file
    xml = aio.fromFile(template_filename)

    # UPDATE_P1: the template xml with the main xml generated here
    # populate basic properties for mesh, regions, and soil properties
    main_xml = ats_input_spec.io.to_xml(main)
    populate_basic_properties(xml, main_xml, **kwargs)

    # UPDATE_P2: runX specific configuration
    # update the start and end time -- start at Oct 1 of year 0, end 10 years later
    # update the start and end time -- would be nice to set these in main, but it would be 
    # confusing as to when to copy them in populate_basic_properties and when not to do so.
    #start_day = 274 # water year
    start_day = 0
    if cyclic_steadystate:
        end_day = start_day + (nyears_cyclic_spinup) * 365 
    else:
        end_day = start_day + (end_year - start_year + 1) * 365 
        
    par = asearch.find_path(xml, ['cycle driver', 'start time'])
    par.setValue(start_day)

    par = asearch.find_path(xml, ['cycle driver', 'end time'])
    par.setValue(end_day)
    
    # update the restart filenames
    for var in asearch.findall_path(xml, ['initial condition', 'restart file']):
        var.setValue(os.path.join('..', outputs[f'{previous}_{name}_rundir'], 'checkpoint_final.h5'))
    print("{previous}_{site_name}_rundir= "+outputs[f'{previous}_{name}_rundir'])

    # update the observations list
    # obs = next(i for (i,el) in enumerate(xml) if el.get('name') == 'observations')
    # xml[obs] = asearch.child_by_name(main_xml, 'observations')

    # P2.2: find and replace the [PKs][subsurface flow][boundary conditions]
    if cyclic_steadystate:
        tmp_BChead_start_filename = os.path.join('..', outputs['BChead_start_spinup_filename_site'])
        tmp_BChead_end_filename = os.path.join('..', outputs['BChead_end_spinup_filename_site'])
    else:
        tmp_BChead_start_filename = os.path.join('..', outputs['BChead_start_transient_filename_site'])
        tmp_BChead_end_filename = os.path.join('..', outputs['BChead_end_transient_filename_site'])
    flow_bc = asearch.find_path(xml, ['PKs', 'subsurface flow', 'boundary conditions'])
    flow_bc.append(hillslope_head_bc_h5file("uphill", "{left_face}", tmp_BChead_start_filename, "Time", "startpt_head",
                                        "outlet", "{right_face}", tmp_BChead_end_filename, "Time", "endpt_head"))
    print(flow_bc)
    
    # P2.3: find and replace the [PKs][surface flow][boundary conditions]
    landflow_bc = asearch.find_path(xml, ['PKs', 'surface flow', 'boundary conditions'])
    landflow_bc.append(hillslope_head_bc_h5file("uphill", "{surface left}", tmp_BChead_start_filename, "Time", "startpt_head",
                                        "outlet", "{surface right}", tmp_BChead_end_filename, "Time", "endpt_head"))
    print(landflow_bc)
      
    # write to disk and make a directory for running the run
    outputs[f'{prefix}_{name}_filename'] = f'../caseflow-{runnum}/{name}_nx{meshsize_nx}_nz{len(dzs_soil)+len(dzs_geo)}.{runnum}.v1.5.xml'
    filename = outputs[f'{prefix}_{name}_filename']

    outputs[f'{prefix}_{name}_rundir'] = f'../caseflow-{runnum}/{name}'
    rundir = outputs[f'{prefix}_{name}_rundir']

    aio.toFile(xml, filename)
    try:
        os.mkdir(rundir)
    except FileExistsError:
        pass


In [ ]:
write_transient(site_name, cyclic_steadystate=True, start_year=start_year_spinup, end_year=end_year_spinup)

In [ ]:
write_transient(site_name, cyclic_steadystate=False, start_year=start_year_transient, end_year=end_year_transient)

# Output Variables Needed by Other Notebooks

In [ ]:
outputs

In [ ]:
# for 2b-add_reaction_cybernetic.ipynb

In [ ]:
# median of soil_thickness is used to convert DOC fluxes in unit gC/m2/s to mol/m3/s
# in ELM_from_huilin/get_docflux_from_ELM_3D.ipynb
print(soil_thickness)
print(outputs['soil_thickness_median'])

In [ ]:
# soil_name is used in 2b-add_reaction_cybernetic.ipynb
# to config the regions for DOC injection
nrcs_set_names = set()
for ats_id in subsurface_props_used.index:
    props = subsurface_props_used.loc[ats_id]
    set_name = soil_set_name(ats_id)
    if props.source == 'NRCS':
        nrcs_set_names.add(set_name)

# join names, then copy paste to 2b-add_reaction_cybernetic.ipynb
output_set_names = ', '.join(nrcs_set_names)
print(output_set_names)

outputs['soil_region_string'] = f'../data-processed/{site_name}/soil_region.txt'
with open(outputs['soil_region_string'], 'w') as f:
    f.write(output_set_names)